<a href="https://colab.research.google.com/github/Arnob07Mondal/Compiler_Design/blob/main/compilerDesignLab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Write a Python Program to find FIRST of a Grammar.(User Defined Input Grammar)**

In [2]:
def find_first(symbol, grammar, first, visited):
    if symbol not in grammar:
        return {symbol}
    if symbol in visited:
        return first[symbol]
    visited.add(symbol)
    for production in grammar[symbol]:
        if production == '#':
            first[symbol].add('#')
            continue
        all_epsilon = True
        for sym in production:
            temp = find_first(sym, grammar, first, visited)
            first[symbol].update(temp - {'#'})
            if '#' not in temp:
                all_epsilon = False
                break
        if all_epsilon:
            first[symbol].add('#')
    return first[symbol]

grammar = {}
n = int(input("Enter number of productions: "))
print("\nEnter productions in the form A->aB|#")
for i in range(n):
    production = input("Production: ")
    lhs, rhs = production.split("->")
    alternatives = rhs.split("|")
    grammar[lhs.strip()] = [
        alt.strip() for alt in alternatives
    ]

first = {non_terminal: set() for non_terminal in grammar}
for non_terminal in grammar:
    find_first(non_terminal, grammar, first, set())

print("\nFIRST Sets:")
for non_terminal in grammar:
    print("FIRST(" + non_terminal + ") = {", end=" ")
    for symbol in sorted(first[non_terminal]):
        print(symbol, end=" ")
    print("}")

Enter number of productions: 3

Enter productions in the form A->aB|#
Production: E->TR
Production: R->TR|#
Production: T->i

FIRST Sets:
FIRST(E) = { i }
FIRST(R) = { # i }
FIRST(T) = { i }


**Write a Python Program to find FOLLOW set from a grammer. (User Defiend Input). **

In [3]:
def first_of_string(symbols, grammar, first):
    result = set()
    for symbol in symbols:
        if symbol not in grammar:
            result.add(symbol)
            return result
        result.update(first[symbol] - {'#'})
        if '#' not in first[symbol]:
            return result
    result.add('#')
    return result

def calculate_first(grammar):
    first = {nt: set() for nt in grammar}
    changed = True
    while changed:
        changed = False
        for nt in grammar:
            for production in grammar[nt]:
                if production == '#':
                    if '#' not in first[nt]:
                        first[nt].add('#')
                        changed = True
                    continue
                for symbol in production:
                    if symbol not in grammar:
                        if symbol not in first[nt]:
                            first[nt].add(symbol)
                            changed = True
                        break
                    old_size = len(first[nt])
                    first[nt].update(first[symbol] - {'#'})
                    if len(first[nt]) != old_size:
                        changed = True
                    if '#' not in first[symbol]:
                        break
                else:
                    if '#' not in first[nt]:
                        first[nt].add('#')
                        changed = True
    return first

def calculate_follow(grammar, first, start_symbol):
    follow = {nt: set() for nt in grammar}

    follow[start_symbol].add('$')

    changed = True

    while changed:
        changed = False

        for nt in grammar:
            for production in grammar[nt]:

                symbols = list(production)

                for i, symbol in enumerate(symbols):

                    if symbol not in grammar:
                        continue

                    beta = symbols[i + 1:]

                    if beta:
                        first_beta = first_of_string(beta, grammar, first)

                        old_size = len(follow[symbol])
                        follow[symbol].update(first_beta - {'#'})

                        if len(follow[symbol]) != old_size:
                            changed = True

                        if '#' in first_beta:
                            old_size = len(follow[symbol])
                            follow[symbol].update(follow[nt])

                            if len(follow[symbol]) != old_size:
                                changed = True

                    else:
                        old_size = len(follow[symbol])
                        follow[symbol].update(follow[nt])

                        if len(follow[symbol]) != old_size:
                            changed = True
    return follow

grammar = {}
n = int(input("Enter number of productions: "))
print("\nEnter productions in the form A->aB|#")
for i in range(n):
    production = input("Production: ")
    lhs, rhs = production.split("->")
    lhs = lhs.strip()
    alternatives = rhs.split("|")
    grammar[lhs] = [alt.strip() for alt in alternatives]

start_symbol = list(grammar.keys())[0]
first = calculate_first(grammar)
follow = calculate_follow(grammar, first, start_symbol)

print("\nFOLLOW Sets:")
for nt in grammar:
    print("FOLLOW(" + nt + ") = {", end=" ")
    for symbol in sorted(follow[nt]):
        print(symbol, end=" ")
    print("}")

Enter number of productions: 4

Enter productions in the form A->aB|#
Production: E->TR
Production: R->+TR|#
Production: T->i
Production: T->(E)

FOLLOW Sets:
FOLLOW(E) = { $ ) }
FOLLOW(R) = { $ ) }
FOLLOW(T) = { $ ) + }
